# FASEROH GSoC 2026 — Taylor Expansion via Seq2Seq
**Task:** Given a symbolic function, predict its 4th-order Maclaurin (Taylor) series.

**Pipeline:**
1. Dataset generation with SymPy → tokenisation
2. LSTM Seq2Seq with Bahdanau attention
3. Transformer Seq2Seq (PyTorch `nn.Transformer`)
4. Evaluation: Exact-Match · Token-Accuracy · BLEU-4 · Numerical R²

> **Run on Google Colab (GPU runtime recommended).  
> Execute cells top-to-bottom.**

In [ ]:
# CELL 1: Install extra deps (only needed on Colab / fresh env)
# scikit-learn for R², nltk for BLEU.  sympy & torch ship with Colab already.
!pip install nltk scikit-learn --quiet

In [ ]:
# CELL 2: Imports & global seeds
import sympy as sp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random, re, json
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('punkt', quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1 · Dataset Generation (SymPy)

In [ ]:
# CELL 3: Generate (function, Taylor-expansion) pairs with SymPy
# Strategy: build a diverse library of single + composite expressions,
# then compute the 4th-order Maclaurin series with sympy.series().

x = sp.Symbol('x')

def taylor(expr, order=4):
    """Return 4th-order Maclaurin series as a sympy expression (O-term removed)."""
    try:
        s = sp.series(expr, x, 0, order + 1).removeO()
        return sp.expand(s)
    except Exception:
        return None

def expr_to_str(e):
    """Canonical string rep of a sympy expression."""
    return str(sp.expand(e))

def generate_dataset(n_target=6000, seed=42):
    random.seed(seed); np.random.seed(seed)
    pairs   = []        # list of (func_str, taylor_str)
    seen    = set()

    def try_add(expr):
        f_str = expr_to_str(expr)
        t_exp = taylor(expr, 4)
        if t_exp is None: return
        t_str = expr_to_str(t_exp)
        # Skip trivially zero expansions or ones we've seen
        if t_str == '0' or f_str in seen: return
        seen.add(f_str)
        pairs.append((f_str, t_str))

    # Base elementary functions
    base = [
        x, x**2, x**3, x**4,
        sp.sin(x), sp.cos(x), sp.tan(x),
        sp.exp(x), sp.exp(-x),
        sp.log(1 + x),
        sp.sqrt(1 + x),
        1 / (1 + x),
        1 / (1 - x),
        sp.sinh(x), sp.cosh(x),
        sp.atan(x), sp.asin(x),
        sp.sin(2*x), sp.cos(2*x),
        sp.exp(x**2),
        sp.log(1 + x**2),
        sp.sin(x)**2,
        sp.cos(x)**2,
        sp.sin(x)*sp.cos(x),
    ]
    for f in base:
        try_add(f)

    # Scaled: c * f(x)
    coeffs = [sp.Integer(c) for c in [-3,-2,-1,2,3,4]] + [sp.Rational(1,2), sp.Rational(3,2)]
    for f in base[:14]:
        for c in coeffs:
            try_add(c * f)

    # Linear combinations: ±f ± g
    for i in range(len(base)):
        for j in range(i+1, min(i+6, len(base))):
            for s1 in [1,-1]:
                for s2 in [1,-1]:
                    try_add(s1*base[i] + s2*base[j])

    # Products: f * g
    products = [
        (x,        sp.sin(x)),  (x,        sp.cos(x)),
        (x,        sp.exp(x)),  (x**2,     sp.sin(x)),
        (x**2,     sp.cos(x)), (x**2,     sp.exp(x)),
        (sp.sin(x), sp.cos(x)), (sp.sin(x), sp.exp(x)),
        (sp.cos(x), sp.exp(x)), (x,        sp.log(1+x)),
        (x,        1/(1+x)),   (sp.sin(x), sp.log(1+x)),
        (x**2,     sp.log(1+x)),(x,        sp.atan(x)),
        (sp.exp(x), sp.log(1+x)),
    ]
    for f, g in products:
        try_add(f * g)
        try_add(-f * g)

    # Random integer-coefficient polynomials
    rng = np.random.RandomState(seed)
    attempts = 0
    while len(pairs) < n_target and attempts < 50_000:
        attempts += 1
        coeffs_p = rng.randint(-4, 5, size=5)
        if all(c == 0 for c in coeffs_p): continue
        expr = sum(int(c)*x**i for i,c in enumerate(coeffs_p) if c != 0)
        if expr == 0: continue
        try_add(expr)

    # Random combos: c1*f + c2*g + poly term
    func_pool = [sp.sin(x), sp.cos(x), sp.exp(x), sp.log(1+x), 1/(1+x), sp.atan(x)]
    rng2 = np.random.RandomState(seed+1)
    for _ in range(20_000):
        if len(pairs) >= n_target: break
        c1 = rng2.randint(-3,4)
        c2 = rng2.randint(-3,4)
        f  = func_pool[rng2.randint(len(func_pool))]
        g  = func_pool[rng2.randint(len(func_pool))]
        p  = rng2.randint(-2,3)
        try_add(c1*f + c2*g + p*x**2)

    return pairs

print("Generating dataset (this takes ~30 s)...")
dataset = generate_dataset(n_target=6000)
print(f"\nDataset size: {len(dataset):,}")
print("\nSample pairs:")
for i in range(6):
    print(f"  [{i}] f(x)   = {dataset[i][0]}")
    print(f"      T(x)   = {dataset[i][1]}")
    print()

In [ ]:
# CELL 4: Exploratory Data Analysis
from collections import Counter

src_raw = [s for s,_ in dataset]
tgt_raw = [t for _,t in dataset]

# Sequence lengths (chars) before tokenisation
src_clen = [len(s) for s in src_raw]
tgt_clen = [len(t) for t in tgt_raw]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(src_clen, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Input expression length (chars)'); axes[0].set_xlabel('chars')
axes[1].hist(tgt_clen, bins=40, color='tomato',     edgecolor='white')
axes[1].set_title('Taylor expansion length (chars)'); axes[1].set_xlabel('chars')
plt.tight_layout(); plt.show()

print(f"Src len  — mean: {np.mean(src_clen):.1f}  max: {max(src_clen)}")
print(f"Tgt len  — mean: {np.mean(tgt_clen):.1f}  max: {max(tgt_clen)}")

# Most common output tokens (to understand vocabulary)
all_tgt_chars = Counter(''.join(tgt_raw))
print("\nTop 20 chars in Taylor expansions:", dict(all_tgt_chars.most_common(20)))

## 2 · Tokenisation

In [ ]:
# CELL 5: Regex tokeniser + Vocabulary
# Tokens:  function names | ** | operators | multi-digit ints | variable | parens
# Example:  "sin(x) - x**3/6"  →  ['sin','(','x',')','−','x','**','3','/','6']

PAD, SOS, EOS, UNK = '<PAD>', '<SOS>', '<EOS>', '<UNK>'
SPECIALS = [PAD, SOS, EOS, UNK]

_TOK_RE = re.compile(
    r'(sin|cos|tan|exp|log|sqrt|sinh|cosh|tanh|asin|acos|atan'
    r'|\*\*|[+\-*/()]|\d+|x)'
)

def tokenise(expr_str: str):
    """Split expression string into symbol tokens."""
    return _TOK_RE.findall(expr_str)


class Vocabulary:
    """Bidirectional token ↔ index mapping with PAD/SOS/EOS/UNK."""
    def __init__(self):
        self.tok2idx = {t: i for i, t in enumerate(SPECIALS)}
        self.idx2tok = {i: t for t, i in self.tok2idx.items()}

    def build(self, token_seqs):
        for seq in token_seqs:
            for tok in seq:
                if tok not in self.tok2idx:
                    idx = len(self.tok2idx)
                    self.tok2idx[tok] = idx
                    self.idx2tok[idx] = tok

    def encode(self, tokens, sos=True, eos=True):
        ids = ([self.tok2idx[SOS]] if sos else [])
        ids += [self.tok2idx.get(t, self.tok2idx[UNK]) for t in tokens]
        ids += ([self.tok2idx[EOS]] if eos else [])
        return ids

    def decode(self, ids, strip_special=True):
        out = []
        for i in ids:
            tok = self.idx2tok.get(int(i), UNK)
            if strip_special and tok in SPECIALS: continue
            out.append(tok)
        return out

    def ids_to_str(self, ids):
        return ''.join(self.decode(ids))

    def __len__(self): return len(self.tok2idx)


# Tokenise everything
src_tok_all = [tokenise(s) for s in src_raw]
tgt_tok_all = [tokenise(t) for t in tgt_raw]

vocab = Vocabulary()
vocab.build(src_tok_all + tgt_tok_all)
VOCAB_SIZE = len(vocab)

src_tok_lens = [len(s) for s in src_tok_all]
tgt_tok_lens = [len(t) for t in tgt_tok_all]
MAX_SRC = max(src_tok_lens) + 2   # +SOS+EOS
MAX_TGT = max(tgt_tok_lens) + 2

print(f"Vocabulary size : {VOCAB_SIZE}")
print(f"Tokens          : {sorted(vocab.tok2idx.keys())}")
print(f"Max src tokens  : {MAX_SRC}")
print(f"Max tgt tokens  : {MAX_TGT}")
print()

# Quick sanity check
sample = "sin(x) + x**2/2 - 1"
toks   = tokenise(sample)
enc    = vocab.encode(toks)
dec    = vocab.ids_to_str(enc)
print(f"Tokenise  '{sample}'")
print(f"  → tokens : {toks}")
print(f"  → ids    : {enc}")
print(f"  → decoded: '{dec}' (should match input)")

In [ ]:
# CELL 6: PyTorch Dataset, padding collate, train/val/test split
class TaylorDataset(Dataset):
    def __init__(self, pairs, max_src=MAX_SRC, max_tgt=MAX_TGT):
        self.samples = []
        for src_str, tgt_str in pairs:
            s = vocab.encode(tokenise(src_str))[:max_src]
            t = vocab.encode(tokenise(tgt_str))[:max_tgt]
            self.samples.append((s, t))

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def collate(batch):
    """Pad within-batch to the longest sequence."""
    src_b, tgt_b = zip(*batch)
    S = max(len(s) for s in src_b)
    T = max(len(t) for t in tgt_b)
    src_pad = torch.zeros(len(src_b), S, dtype=torch.long)
    tgt_pad = torch.zeros(len(tgt_b), T, dtype=torch.long)
    for i,(s,t) in enumerate(zip(src_b,tgt_b)):
        src_pad[i,:len(s)] = torch.tensor(s)
        tgt_pad[i,:len(t)] = torch.tensor(t)
    return src_pad, tgt_pad


# 70 / 15 / 15 split
random.shuffle(dataset)
n = len(dataset)
cut1, cut2 = int(.70*n), int(.85*n)
train_data, val_data, test_data = dataset[:cut1], dataset[cut1:cut2], dataset[cut2:]

BATCH = 64
train_loader = DataLoader(TaylorDataset(train_data), batch_size=BATCH, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(TaylorDataset(val_data),   batch_size=BATCH, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(TaylorDataset(test_data),  batch_size=BATCH, shuffle=False, collate_fn=collate)

print(f"Train: {len(train_data):,}  Val: {len(val_data):,}  Test: {len(test_data):,}")
print(f"Batches per epoch (train): {len(train_loader)}")

In [ ]:
# CELL 7: Shared evaluation helpers
def decode_batch(ids_matrix):
    """ids_matrix: (batch, seq_len) numpy – returns list of reconstructed strings."""
    out = []
    for row in ids_matrix:
        ids = list(row)
        eos_idx = vocab.tok2idx[EOS]
        if eos_idx in ids:
            ids = ids[:ids.index(eos_idx)]
        out.append(vocab.ids_to_str(ids))
    return out


def compute_metrics(preds, trues):
    """
    preds / trues: list of expression strings (reconstructed from token ids).
    Returns dict with exact_match, token_acc, bleu4, r2_numerical.
    """
    smooth = SmoothingFunction().method1

    # Exact match
    exact = sum(p == t for p,t in zip(preds,trues)) / len(preds)

    # Token accuracy
    correct = total = 0
    for p,t in zip(preds,trues):
        pt, tt = tokenise(p), tokenise(t)
        L = min(len(pt), len(tt))
        correct += sum(a==b for a,b in zip(pt[:L], tt[:L]))
        total   += max(len(pt), len(tt))
    tok_acc = correct / total if total > 0 else 0.0

    # BLEU-4
    refs = [[tokenise(t)] for t in trues]
    hyps = [tokenise(p) for p in preds]
    bleu = corpus_bleu(refs, hyps, smoothing_function=smooth)

    # Numerical R² – evaluate polynomials on a grid
    xv   = np.linspace(-0.5, 0.5, 100)
    xs   = sp.Symbol('x')
    r2s  = []
    for p_str, t_str in zip(preds, trues):
        try:
            pf = sp.lambdify(xs, sp.sympify(p_str), 'numpy')
            tf = sp.lambdify(xs, sp.sympify(t_str), 'numpy')
            yp = np.asarray(pf(xv), dtype=float)
            yt = np.asarray(tf(xv), dtype=float)
            if np.any(~np.isfinite(yp)) or np.any(~np.isfinite(yt)): continue
            r2s.append(max(-1.0, r2_score(yt, yp)))
        except Exception:
            r2s.append(0.0)
    r2_mean = float(np.mean(r2s)) if r2s else 0.0

    return dict(exact_match=exact, token_acc=tok_acc, bleu4=bleu, r2=r2_mean)


def print_metrics(name, m):
    print(f"\n{'='*50}")
    print(f"  {name} — TEST METRICS")
    print(f"{'='*50}")
    print(f"  Exact Match    : {m['exact_match']:.4f}")
    print(f"  Token Accuracy : {m['token_acc']:.4f}")
    print(f"  BLEU-4         : {m['bleu4']:.4f}")
    print(f"  Numerical R²   : {m['r2']:.4f}")
    print(f"{'='*50}")

## 3 · LSTM Seq2Seq with Bahdanau Attention

In [ ]:
# CELL 8: LSTM Encoder / Decoder / Seq2Seq model
class BahdanauAttention(nn.Module):
    """Additive (Bahdanau) attention over encoder outputs."""
    def __init__(self, enc_hid, dec_hid):
        super().__init__()
        self.W_enc = nn.Linear(enc_hid, dec_hid, bias=False)
        self.W_dec = nn.Linear(dec_hid, dec_hid, bias=False)
        self.v     = nn.Linear(dec_hid, 1, bias=False)

    def forward(self, enc_out, dec_h):
        # enc_out: (B, S, enc_hid)   dec_h: (B, dec_hid)
        e = torch.tanh(self.W_enc(enc_out) + self.W_dec(dec_h).unsqueeze(1))
        return F.softmax(self.v(e).squeeze(-1), dim=1)   # (B, S)


class LSTMEncoder(nn.Module):
    def __init__(self, vocab_sz, emb_dim, hid, n_layers, dropout):
        super().__init__()
        self.emb  = nn.Embedding(vocab_sz, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hid, n_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.fc_h = nn.Linear(hid*2, hid)
        self.fc_c = nn.Linear(hid*2, hid)
        self.drop = nn.Dropout(dropout)
        self.n_layers = n_layers

    def forward(self, src):
        emb = self.drop(self.emb(src))                   # (B,S,emb)
        out, (h, c) = self.lstm(emb)                     # out:(B,S,hid*2)
        # Merge top fwd+bwd layer → init decoder state
        h_top = torch.tanh(self.fc_h(torch.cat([h[-2],h[-1]],1)))
        c_top = torch.tanh(self.fc_c(torch.cat([c[-2],c[-1]],1)))
        h = h_top.unsqueeze(0).expand(self.n_layers,-1,-1).contiguous()
        c = c_top.unsqueeze(0).expand(self.n_layers,-1,-1).contiguous()
        return out, h, c


class LSTMDecoder(nn.Module):
    def __init__(self, vocab_sz, emb_dim, hid, n_layers, dropout):
        super().__init__()
        self.emb  = nn.Embedding(vocab_sz, emb_dim, padding_idx=0)
        self.attn = BahdanauAttention(hid*2, hid)
        self.lstm = nn.LSTM(emb_dim + hid*2, hid, n_layers,
                            batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hid + hid*2 + emb_dim, vocab_sz)
        self.drop = nn.Dropout(dropout)

    def step(self, tok, h, c, enc_out):
        emb = self.drop(self.emb(tok.unsqueeze(1)))         # (B,1,emb)
        w   = self.attn(enc_out, h[-1])                      # (B,S)
        ctx = torch.bmm(w.unsqueeze(1), enc_out)             # (B,1,hid*2)
        out, (h,c) = self.lstm(torch.cat([emb,ctx],2), (h,c))
        logit = self.fc(torch.cat([out,ctx,emb],2).squeeze(1))
        return logit, h, c


class LSTMSeq2Seq(nn.Module):
    def __init__(self, vocab_sz, emb_dim=128, hid=256, n_layers=2, dropout=0.30):
        super().__init__()
        self.enc = LSTMEncoder(vocab_sz, emb_dim, hid, n_layers, dropout)
        self.dec = LSTMDecoder(vocab_sz, emb_dim, hid, n_layers, dropout)
        self.sos = vocab.tok2idx[SOS]
        self.eos = vocab.tok2idx[EOS]

    def forward(self, src, tgt, tf_ratio=0.50):
        B, T   = tgt.shape
        V      = self.dec.fc.out_features
        enc_out, h, c = self.enc(src)
        outputs = torch.zeros(B, T, V, device=src.device)
        tok     = tgt[:,0]                               # always SOS
        for t in range(1, T):
            logit, h, c = self.dec.step(tok, h, c, enc_out)
            outputs[:,t] = logit
            tok = tgt[:,t] if random.random() < tf_ratio else logit.argmax(1)
        return outputs

    @torch.no_grad()
    def generate(self, src, max_len=MAX_TGT):
        self.eval()
        B = src.shape[0]
        enc_out, h, c = self.enc(src)
        tok = torch.full((B,), self.sos, dtype=torch.long, device=src.device)
        seqs = []
        for _ in range(max_len):
            logit, h, c = self.dec.step(tok, h, c, enc_out)
            tok = logit.argmax(1)
            seqs.append(tok.cpu().numpy())
            if (tok == self.eos).all(): break
        return np.column_stack(seqs) if seqs else np.zeros((B,1),int)


lstm_model = LSTMSeq2Seq(VOCAB_SIZE).to(device)
n_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f"LSTM Seq2Seq — trainable parameters: {n_params:,}")

In [ ]:
# CELL 9: LSTM training loop
# Teacher-forcing ratio decays linearly from 0.50 → 0.05 over training.

N_EPOCHS   = 30
LR_LSTM    = 3e-4

lstm_optim = torch.optim.Adam(lstm_model.parameters(), lr=LR_LSTM)
lstm_crit  = nn.CrossEntropyLoss(ignore_index=0)  # ignore PAD
lstm_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(lstm_optim, patience=4, factor=0.5)

lstm_train_hist, lstm_val_hist = [], []


def run_epoch_lstm(model, loader, optim, crit, train, tf_ratio=0.0):
    model.train(train)
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            out = model(src, tgt, tf_ratio=tf_ratio if train else 0.0)
            # shift: predict positions 1..T from 0..T-1
            loss = crit(out[:,1:].reshape(-1,out.shape[-1]), tgt[:,1:].reshape(-1))
            if train:
                optim.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optim.step()
            total_loss += loss.item()
    return total_loss / len(loader)


print("Training LSTM …")
best_val = float('inf')
for ep in range(1, N_EPOCHS+1):
    tf = max(0.05, 0.50 - 0.45*(ep/N_EPOCHS))   # linear decay
    tr_loss = run_epoch_lstm(lstm_model, train_loader, lstm_optim, lstm_crit, train=True,  tf_ratio=tf)
    va_loss = run_epoch_lstm(lstm_model, val_loader,   lstm_optim, lstm_crit, train=False)
    lstm_sched.step(va_loss)
    lstm_train_hist.append(tr_loss)
    lstm_val_hist.append(va_loss)
    if ep % 5 == 0 or ep == 1:
        print(f"  Ep {ep:3d}/{N_EPOCHS}  train={tr_loss:.4f}  val={va_loss:.4f}  tf={tf:.2f}")
    if va_loss < best_val:
        best_val = va_loss
        torch.save(lstm_model.state_dict(), '/tmp/lstm_best.pt')

# Load best checkpoint
lstm_model.load_state_dict(torch.load('/tmp/lstm_best.pt'))
print(f"\nBest val loss: {best_val:.4f}")

In [ ]:
# CELL 10: LSTM evaluation on test set
def infer(model, loader):
    """Collect greedy predictions and ground-truth strings for full loader."""
    all_preds, all_trues = [], []
    model.eval()
    for src, tgt in loader:
        src = src.to(device)
        preds = model.generate(src)            # (B, seq_len)
        all_preds.extend(decode_batch(preds))
        all_trues.extend(decode_batch(tgt.numpy()))
    return all_preds, all_trues

lstm_preds, lstm_trues = infer(lstm_model, test_loader)
lstm_m = compute_metrics(lstm_preds, lstm_trues)
print_metrics("LSTM", lstm_m)

print("\nSample predictions (LSTM):")
for i in range(6):
    marker = "✓" if lstm_preds[i] == lstm_trues[i] else "✗"
    print(f"  {marker} true : {lstm_trues[i]}")
    print(f"    pred : {lstm_preds[i]}")
    print()

## 4 · Transformer Seq2Seq

In [ ]:
# CELL 11: Transformer Seq2Seq model
class SinCosPositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding."""
    def __init__(self, d_model, dropout=0.10, max_len=512):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.)/d_model))
        pe[:,0::2] = torch.sin(pos * div)
        pe[:,1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))   # (1, max_len, d)

    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])


class TransformerSeq2Seq(nn.Module):
    def __init__(self, vocab_sz, d_model=128, nhead=4,
                 n_enc=3, n_dec=3, d_ff=512, dropout=0.10):
        super().__init__()
        self.d_model = d_model
        self.src_emb = nn.Embedding(vocab_sz, d_model, padding_idx=0)
        self.tgt_emb = nn.Embedding(vocab_sz, d_model, padding_idx=0)
        self.pe      = SinCosPositionalEncoding(d_model, dropout)
        self.tf      = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=n_enc, num_decoder_layers=n_dec,
            dim_feedforward=d_ff, dropout=dropout, batch_first=True
        )
        self.out_proj = nn.Linear(d_model, vocab_sz)
        self._init()

    def _init(self):
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)

    # Masks
    def _pad_mask(self, seq):
        return seq == 0                         # True = ignore

    def _causal_mask(self, sz):
        return torch.triu(torch.ones(sz,sz,device=next(self.parameters()).device),1).bool()

    # Forward (teacher-forcing)
    def forward(self, src, tgt):
        scale = self.d_model ** 0.5
        src_e = self.pe(self.src_emb(src) * scale)
        tgt_e = self.pe(self.tgt_emb(tgt) * scale)
        out   = self.tf(
            src_e, tgt_e,
            tgt_mask               = self._causal_mask(tgt.size(1)),
            src_key_padding_mask   = self._pad_mask(src),
            tgt_key_padding_mask   = self._pad_mask(tgt),
            memory_key_padding_mask= self._pad_mask(src),
        )
        return self.out_proj(out)               # (B, T, vocab)

    # Autoregressive generation
    @torch.no_grad()
    def generate(self, src, max_len=MAX_TGT):
        self.eval()
        B   = src.shape[0]
        sos = vocab.tok2idx[SOS]
        eos = vocab.tok2idx[EOS]
        gen = torch.full((B,1), sos, dtype=torch.long, device=src.device)
        for _ in range(max_len):
            logits = self.forward(src, gen)
            nxt    = logits[:,-1,:].argmax(-1, keepdim=True)
            gen    = torch.cat([gen, nxt], dim=1)
            if (nxt.squeeze(1) == eos).all(): break
        return gen[:,1:].cpu().numpy()          # strip SOS


trans_model = TransformerSeq2Seq(VOCAB_SIZE).to(device)
n_params_t  = sum(p.numel() for p in trans_model.parameters() if p.requires_grad)
print(f"Transformer Seq2Seq — trainable parameters: {n_params_t:,}")

In [ ]:
# CELL 12: Transformer training loop
# Uses label-smoothed cross-entropy + cosine LR schedule.

T_EPOCHS = 30
LR_TRANS = 5e-4

t_optim = torch.optim.Adam(trans_model.parameters(), lr=LR_TRANS, betas=(0.9,0.98), eps=1e-9)
t_crit  = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.10)
t_sched = torch.optim.lr_scheduler.CosineAnnealingLR(t_optim, T_max=T_EPOCHS, eta_min=1e-5)

trans_train_hist, trans_val_hist = [], []


def run_epoch_trans(model, loader, optim, crit, train):
    model.train(train)
    total = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_in  = tgt[:,:-1]      # feed SOS … last-1
            tgt_out = tgt[:,1:]       # predict 1 … EOS
            logits  = model(src, tgt_in)
            loss    = crit(logits.reshape(-1,logits.shape[-1]), tgt_out.reshape(-1))
            if train:
                optim.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optim.step()
            total += loss.item()
    return total / len(loader)


print("Training Transformer …")
best_val_t = float('inf')
for ep in range(1, T_EPOCHS+1):
    tr = run_epoch_trans(trans_model, train_loader, t_optim, t_crit, train=True)
    va = run_epoch_trans(trans_model, val_loader,   t_optim, t_crit, train=False)
    t_sched.step()
    trans_train_hist.append(tr)
    trans_val_hist.append(va)
    if ep % 5 == 0 or ep == 1:
        print(f"  Ep {ep:3d}/{T_EPOCHS}  train={tr:.4f}  val={va:.4f}")
    if va < best_val_t:
        best_val_t = va
        torch.save(trans_model.state_dict(), '/tmp/trans_best.pt')

trans_model.load_state_dict(torch.load('/tmp/trans_best.pt'))
print(f"\nBest val loss: {best_val_t:.4f}")

In [ ]:
# CELL 13: Transformer evaluation on test set
def infer_trans(model, loader):
    all_preds, all_trues = [], []
    model.eval()
    for src, tgt in loader:
        src = src.to(device)
        preds = model.generate(src)
        all_preds.extend(decode_batch(preds))
        all_trues.extend(decode_batch(tgt.numpy()))
    return all_preds, all_trues

trans_preds, trans_trues = infer_trans(trans_model, test_loader)
trans_m = compute_metrics(trans_preds, trans_trues)
print_metrics("Transformer", trans_m)

print("\nSample predictions (Transformer):")
for i in range(6):
    marker = "✓" if trans_preds[i] == trans_trues[i] else "✗"
    print(f"  {marker} true : {trans_trues[i]}")
    print(f"    pred : {trans_preds[i]}")
    print()

## 5 · Results & Comparison

In [ ]:
# CELL 14: Side-by-side plots + summary table
fig = plt.figure(figsize=(16, 11))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)

EPOCHS = range(1, N_EPOCHS+1)

# (A) Training loss curves
ax0 = fig.add_subplot(gs[0,:])
ax0.plot(EPOCHS, lstm_train_hist,  'steelblue',  lw=1.8, label='LSTM – train')
ax0.plot(EPOCHS, lstm_val_hist,    'steelblue',  lw=1.8, ls='--', alpha=0.7, label='LSTM – val')
ax0.plot(EPOCHS, trans_train_hist, 'tomato',     lw=1.8, label='Transformer – train')
ax0.plot(EPOCHS, trans_val_hist,   'tomato',     lw=1.8, ls='--', alpha=0.7, label='Transformer – val')
ax0.set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
ax0.set_xlabel('Epoch'); ax0.set_ylabel('Cross-Entropy Loss')
ax0.legend(); ax0.grid(alpha=0.3)

# (B) Metrics bar chart
ax1 = fig.add_subplot(gs[1,0])
labels = ['Exact\nMatch', 'Token\nAcc', 'BLEU-4', 'Numerical\nR²']
lv = [lstm_m['exact_match'],  lstm_m['token_acc'],  lstm_m['bleu4'],  lstm_m['r2']]
tv = [trans_m['exact_match'], trans_m['token_acc'], trans_m['bleu4'], trans_m['r2']]
x  = np.arange(len(labels)); w = 0.35
b1 = ax1.bar(x-w/2, lv, w, label='LSTM',        color='steelblue', alpha=0.85)
b2 = ax1.bar(x+w/2, tv, w, label='Transformer', color='tomato',    alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels(labels, fontsize=9)
ax1.set_ylim(0, 1.12); ax1.set_title('Test Metrics', fontsize=12, fontweight='bold')
ax1.legend(); ax1.grid(alpha=0.3, axis='y')
for bar in list(b1)+list(b2):
    h = bar.get_height()
    ax1.annotate(f'{h:.3f}', xy=(bar.get_x()+bar.get_width()/2, h),
                 xytext=(0,3), textcoords='offset points', ha='center', fontsize=7)

# (C) Exact match scatter: true vs predicted numeric value at x=0.25
ax2 = fig.add_subplot(gs[1,1])
xs_sym = sp.Symbol('x')
pt_true, pt_pred_lstm, pt_pred_trans = [], [], []
for t, lp, tp in zip(trans_trues, lstm_preds, trans_preds):
    try:
        yt = float(sp.sympify(t).subs(xs_sym, 0.25))
        yl = float(sp.sympify(lp).subs(xs_sym, 0.25))
        yt2= float(sp.sympify(tp).subs(xs_sym, 0.25))
        if all(abs(v)<50 for v in [yt,yl,yt2]):
            pt_true.append(yt); pt_pred_lstm.append(yl); pt_pred_trans.append(yt2)
    except: pass

lim = max(abs(min(pt_true+pt_pred_trans)), abs(max(pt_true+pt_pred_trans))) + 0.5
ax2.scatter(pt_true, pt_pred_lstm,  s=10, alpha=0.4, c='steelblue', label='LSTM')
ax2.scatter(pt_true, pt_pred_trans, s=10, alpha=0.4, c='tomato',    label='Transformer')
ax2.plot([-lim,lim],[-lim,lim],'k--', lw=1, label='perfect')
ax2.set_xlim(-lim,lim); ax2.set_ylim(-lim,lim)
ax2.set_xlabel('True f(0.25)'); ax2.set_ylabel('Predicted f(0.25)')
ax2.set_title('Numerical Value at x=0.25', fontsize=12, fontweight='bold')
ax2.legend(markerscale=2); ax2.grid(alpha=0.3)

plt.savefig('faseroh_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f"\n{'-'*56}")
print(f"{'Metric':<22}{'LSTM':>15}{'Transformer':>15}")
print(f"{'-'*56}")
for k,label in [('exact_match','Exact Match'),('token_acc','Token Accuracy'),
                ('bleu4','BLEU-4'),('r2','Numerical R²')]:
    print(f"  {label:<20}{lstm_m[k]:>15.4f}{trans_m[k]:>15.4f}")
print(f"{'-'*56}")
print(f"  {'Params':<20}{sum(p.numel() for p in lstm_model.parameters()):>15,}{sum(p.numel() for p in trans_model.parameters()):>15,}")
print(f"{'-'*56}")